# Data Understanding

I wish I could tell you that your data will always be clean, complete, and ready for analysis.  I really do.  The truth however is far from it.  Very rarely does data come ready to go for any sort of analytics.  It's only after we go through the effort to identify, extract, and play with and massage the data to prove there's some value in the solution we're building, that we then go to the next step of building a standard data asset.  Once created, this new data asset allows us to deliver the data in exactly the way we need it for use in our production program from now on.  But we have to get there first through exploratory analysis.

There's no end to the unique challenges you'll run into once you start working with data, and again there's no standardization from data set to data set.  What we can do though is lay out a template workflow that will always be a good idea to start with, independent of any differences you run across working with your various datasets.

This section is as straight forward as it sounds.  We simply want to begin getting a sense for our data.

> _Did the business send us what we were expecting?_ <br>
> _Do we understand what we have?_
 
Here we'll go through the following three sections in this lesson using a real-world datset.

1. Datset Type
2. Data Structure
3. Data Shape

**<h3>Dataset Type</h3>**

First things first.  Almost without exception, the first thing we want to do is simply take a look at the data.  Let's just see what we have.  

The data we'll play with in this section is a large sleep study tracking the average minutes per night for various sleep patterns, over a one week period of time, for 877 participants.  The recorded sleep states are as follows.

- **active_mins** = not asleep
- **sleep_disturb_mins** =  sleeping, but not in a REM cycle
- **sleep_rem_mins** = sleeping, REM cycle

In [63]:
import pandas as pd

# Read in the data from github repository
url = 'https://github.com/bradybr/practical-data-science-and-ml/blob/main/datasets/sleep_study.csv?raw=true'
dat = pd.read_csv(url, sep = ',')
dat

,id,gender,age,country,study_begin,study_end,active_mins,sleep_disturb_mins,sleep_rem_mins
0,1,F,38.0,USA,4/3/2021,4/9/2023,96.8,227.5,25.9
1,2,M,72.0,Poland,4/3/2021,4/9/2023,245.6,644.2,86.7
2,3,F,95.0,Italy,4/3/2021,4/9/2023,279.4,465.6,31.8
3,4,M,37.0,USA,4/3/2021,4/9/2023,60.0,109.0,19.7
4,5,F,80.0,Spain,4/3/2021,4/9/2023,89.4,113.3,38.6
...,...,...,...,...,...,...,...,...,...
872,872,F,17.0,Italy,4/3/2021,4/9/2023,42.1,87.7,42.3
873,873,F,64.0,USA,4/3/2021,4/9/2023,128.5,268.8,23.4
874,874,F,27.0,Italy,4/3/2021,4/9/2023,43.2,84.1,18.0
875,875,F,69.0,Spain,4/3/2021,4/9/2023,246.8,237.6,44.1


What can we say about the type of dataset we're dealing with?  Seems plain enough we're dealing with a _cross-sectional_ dataset.  It appears to be one person per row as our unit of analysis, and we don't see any repeating time periods or values recorded at various intervals.  Should just be one record per person.

**<h3>Data Structure</h3>**

Recall with data structures, we simply want to understand how the data and subjects relate to each other in the dataset.  Here we're looking to understand what's in a row of data.  This is generally a couple of things we want to check.  The 1) Unit/Levels of Analysis and 2) if we have Wide or Long data, and lastly 3) if there's anything peculiar with how the data is represented that we need to be aware of.

Let's go through them one by one.

<h4>Unit of Analysis</h4>

I'm afraid I've already spilled the beans on this one and said I believe it's

> Sleep pattern minutes, by **participant (id)** <br>

At a cursory glance, it looks like there's one person per row as our level of analysis.  Each participant should have a unique "id" that identifies them, then some demographic details, and finally their average minutes for each sleep state during the study.

We'll discuss this in a more methodical way soon when we get to your data quality report, but for now here's a very common check I like to perform straight away to see if I might have any issues.  If I think I should have one record per person because I know the unit of analysis is **by id**, I'll run a quick count by id to see how many rows (records) each has in the data.  It's a quick way to see if what you think is really going on is actually what you have.

Let's try.

Below is the output of counts by the unique number of records for all patients observed in the dataset.  There are 873 patients who have just 1 record, and there are 2 patients with 2 records associated with their patient ids.

In [54]:
id_cts = dat.groupby('id')['id'].size().to_list()
pd.Series(id_cts).value_counts().to_dict()

{1: 873, 2: 2}

We can learn two things from this simple check and the output of counts above.  1) Our level of analysis of **by participant id** is most likely correct, and 2) we have two people who have 2 records associated with their ids that we'll need to deal with at some point.  We'll address fixing the issue when we get to the {doc}`preprocessing` section so sit tight.

<h4>Wide or Long Data</h4>

This one isn't such a huge deal at the moment, but it's a good idea just to take notice if you can.  Evenutally it'll come into play when we begin looking at plotting our data and thinking about reshaping it.  Certain analyses and methodologies require the data to be in certain shapes, so it's nice to know up front what you're dealing with.

In [56]:
dat

,id,gender,age,country,study_begin,study_end,active_mins,sleep_disturb_mins,sleep_rem_mins
0,1,F,38.0,USA,4/3/2021,4/9/2023,96.8,227.5,25.9
1,2,M,72.0,Poland,4/3/2021,4/9/2023,245.6,644.2,86.7
2,3,F,95.0,Italy,4/3/2021,4/9/2023,279.4,465.6,31.8
3,4,M,37.0,USA,4/3/2021,4/9/2023,60.0,109.0,19.7
4,5,F,80.0,Spain,4/3/2021,4/9/2023,89.4,113.3,38.6
...,...,...,...,...,...,...,...,...,...
872,872,F,17.0,Italy,4/3/2021,4/9/2023,42.1,87.7,42.3
873,873,F,64.0,USA,4/3/2021,4/9/2023,128.5,268.8,23.4
874,874,F,27.0,Italy,4/3/2021,4/9/2023,43.2,84.1,18.0
875,875,F,69.0,Spain,4/3/2021,4/9/2023,246.8,237.6,44.1


What do you think?  Hopefully you said _wide_.  There's really not much to this one, and it's usually pretty easy to figure after you have your unit of analysis understand.

So really not much to say with this one.  We have one record per person and our features are spread out individually across the columns.  Easy enough.

Let's move on.

**<h3>Data Shape</h3>**

````{margin}
```{note}
**Date shape** refers to the number of rows (observations) and columns (features/variables) in a dataset.
```
````

What do we mean by data shape?  If you haven't picked up on it yet, **data shape** refers to the number of rows (observations) and columns (features/variables) in the data.  Yep, that's it.  It's simple, but super important for a number of reasons.

For one, does it match what you were expecting?  If the business told us we should expect a dataset with rough 10,000 examples of something occurring, but the dataset we received only has 100 rows, well then you know right away something's probably wrong.  If they told us we should have around 20 different features to use to try and predict some occurence, but we only see 2 columns in the data... you guessed it.  Something's wrong.  

Another reason we care about the structure of our data is because it will change how our subjects and conditioning variables are represented.  If we should have 20 independent features and our data is in a _long_ format, then you won't have 20+ columns of data.  You should be looking for a few columns which have those 20 features spread out vertically instead of horizontally.  Starting to understand how all of these topics fit together?!?!?

```{tip}
Remember, we're constantly looking to validate that what we have or what we've done is behaving as expected.
```

<h4>Observations</h4>

Observations are a foundational topic in the world of statistics.  Here, we're simply talking about the number of records in our data relating to the subject, in this case participant id.

For our data table print out above, at the bottom it shows us the number of rows (observations) and columns (features/variables) are 877 and 9, respectively.  `877 rows × 9 columns`  Does that make sense for what we expect?  877 total participants in the study?  To answer this question we'd need to check with our business partner, but for now let's say we did that and this is approximately what we understood would be in the data.

You should already be noticing an issue hopefully...  Recall that we just looked into how many records were associated with each patient id?  We saw there were 873 patients with one row and 2 people had multiple rows.  Remember?  Well that means we actually only have 875 total patients, not 877!  Here's our first watchout that the total number of rows may not reflect the unit of analysis properly.

This may not matter so much when it's a study of anonymous patients and we have 2 less than we expected.  Imagine if we're trying to analyze production volume from our 10 manufacturing plants though, and we find out we only have 8 represented in the data but couldn't see it because we only looked at the total number of rows in our data?  In this case we might be missing upwards of a 1/3 of our volume, depending on which plants they were!

The last consideration with the number of observations goes back to the reference made to statistics.  Much of what we do with machine learning relies heavily on assumptions made regarding the distribution of the data and the associated statistical moments (e.g. mean, standard deviation).  We'll cover this in more detail later, but for now understand that the trustworthiness of these assumptions falls apart when we have small sample sizes.  The relationship between the number of observations (samples), the number of features (variables), and the number of estimated parameters of a model, are enormously important concerns we'll definitely cover soon.

For now we simply want to get a sense of how many samples and observations we have for whatever our unit of analysis is.

<h4>Features</h4>

````{margin}
```{note}
**Features** are the independent and dependent variables in a data set.  In a structured, wide data set (rows and columns), we can think of the features as the columns of data.
```
````

After understanding the general shape and number of observations in our data, the next thing I want to consider are the features.  This one should be pretty easy to understand since it's very similar to data formats and structures most of us are already familiar with using Excel.  In most cases as with wide data, you can think of the **features** as the variables, or columns in the data.  

As usual, we're always looking for anything that looks suspicious or questionable.  

>_Do you have all of the features (variables) you expect to have?_<br>
>_Do the features (variables) have the correct data types (numeric, string, date, etc.)?_

So where do we start?  Whether you're coding yourself or working in an analytics platform, there's going to be an easy print out or wizard that allows you to get to this information easily.  Let's see it in python.

In [58]:
dat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 877 entries, 0 to 876
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  877 non-null    object 
 1   gender              877 non-null    object 
 2   age                 876 non-null    float64
 3   country             877 non-null    object 
 4   study_begin         877 non-null    object 
 5   study_end           864 non-null    object 
 6   active_mins         873 non-null    float64
 7   sleep_disturb_mins  871 non-null    float64
 8   sleep_rem_mins      868 non-null    float64
dtypes: float64(4), object(5)
memory usage: 61.8+ KB


Checking whether we have all of the variables we expected to see is an obvious since we of course can't perform an analysis on a specific variable if it's not included in the data!  We'll discuss some of the more detailed topics like data variable types in the next section, so for now let's say this is exactly what we were expecting and leave it there.

We were told to expect a unique personal identifier as we see with `id`.  We also expected to see several features having to do with demographics like `gender`, `age`, and `country`.  Then following we expected to see the details of the study with the beginning and ending dates in `study_begin` and `study_end`, and finally the actual sleep minutes recorded for each patient under a few different designations of activity that we see in `active_mins`, `sleep_disturb_mins`, and `sleep_rem_mins`.

Seems like we're all good for now with the features we expected to see!

Remember, this was just a quick look at the data to see if it passes the sniff test and lines up with what you expected to receive.  The objective is to quickly get back to your business partners or data engineers who gave you the data if something looks off.  There are many more exploratory and detailed views we will get into next with your Data Quality Report that may point to issues, but this first glance was intended to get us acquainted with what we have, and quickly let us know if we we're way off track from our expectations.

From here we take a deeper look into our data with what's known as a Data Quality Report.